In [1]:
import pandas as pd
import os

# Загрузка обработанных данных
customers = pd.read_csv('../data/processed/customers_with_rfm.csv')
orders = pd.read_csv('../data/processed/orders_clean.csv')
abc_xyz = pd.read_csv('../data/processed/abc_xyz_products.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Безопасный выбор колонок (не упадёт, если колонки нет)
def pick(df, cols):
    return df[[c for c in cols if c in df.columns]].copy()

# ВИТРИНА 1: клиенты (RFM-страница, фильтры)
mart_customers = pick(customers, [
    'customer_id', 'country', 'age', 'gender', 'membership_tier',
    'registration_date', 'recency', 'frequency', 'monetary',
    'R_score', 'F_score', 'M_score', 'RFM_Score', 'Segment', 'churned'])

# ВИТРИНА 2: заказы (детальная страница, все расчёты)
mart_orders = pick(orders, [
    'order_id', 'customer_id', 'order_date', 'year', 'month', 'quarter',
    'category', 'product_name', 'quantity', 'unit_price_usd', 'subtotal_usd',
    'discount_pct', 'discount_amount_usd', 'shipping_fee_usd', 'tax_amount_usd',
    'total_amount_usd', 'order_status', 'payment_method', 'device_used',
    'session_duration_minutes', 'has_rating', 'customer_rating', 'returned'])
mart_orders['order_year_month'] = mart_orders['order_date'].dt.to_period('M').astype(str)
mart_orders['is_delivered'] = (mart_orders['order_status'] == 'Delivered').astype(int)

# ВИТРИНА 3: товары (ABC-страница); return_rate считаем из orders, а не из product_summary
prod_stats = orders.groupby(['category', 'product_name']).agg(
    orders_cnt=('order_id', 'size'),
    revenue_usd=('total_amount_usd', 'sum'),
    returned_cnt=('returned', 'sum'),
    rated_cnt=('has_rating', 'sum'),
    avg_rating=('customer_rating', 'mean')).reset_index()
prod_stats['return_rate_pct'] = (prod_stats['returned_cnt'] / prod_stats['orders_cnt'] * 100).round(2)
prod_stats['avg_rating'] = prod_stats['avg_rating'].round(2)
mart_products = prod_stats.merge(
    abc_xyz[['product_name', 'ABC_Group', 'XYZ_Group']], on='product_name', how='left')
mart_products['revenue_share_pct'] = (
    mart_products['revenue_usd'] / mart_products['revenue_usd'].sum() * 100).round(2)

# ВИТРИНА 4: месяцы (тренды); пересчитано из orders — источника истины
o = orders.copy()
o['ym'] = o['order_date'].dt.to_period('M')
mart_monthly = o.groupby('ym').agg(
    orders_all=('order_id', 'size'),
    orders_delivered=('order_status', lambda s: (s == 'Delivered').sum()),
    revenue_all_usd=('total_amount_usd', 'sum'),
    unique_customers=('customer_id', 'nunique')).reset_index()
delivered_rev = (o[o['order_status'] == 'Delivered']
                 .groupby('ym')['total_amount_usd'].sum()
                 .rename('revenue_delivered_usd'))
mart_monthly = mart_monthly.merge(delivered_rev, on='ym', how='left')
mart_monthly['year'] = mart_monthly['ym'].dt.year
mart_monthly['month'] = mart_monthly['ym'].dt.month
mart_monthly['quarter'] = mart_monthly['ym'].dt.quarter
mart_monthly['aov_delivered_usd'] = (
    mart_monthly['revenue_delivered_usd'] / mart_monthly['orders_delivered']).round(2)
mart_monthly['ym'] = mart_monthly['ym'].astype(str)

# Сохранение
os.makedirs('../data/marts', exist_ok=True)
mart_customers.to_csv('../data/marts/mart_customers.csv', index=False)
mart_orders.to_csv('../data/marts/mart_orders.csv', index=False)
mart_products.to_csv('../data/marts/mart_products.csv', index=False)
mart_monthly.to_csv('../data/marts/mart_monthly.csv', index=False)

for name, df in [('customers', mart_customers), ('orders', mart_orders),
                 ('products', mart_products), ('monthly', mart_monthly)]:
    print(f"mart_{name}: {df.shape}")

mart_customers: (8000, 15)
mart_orders: (25000, 25)
mart_products: (140, 11)
mart_monthly: (75, 10)
